# QLoRA Hands-On Fine-tuning Notebook

Actually **fine-tune a small LLM (TinyLlama-1.1B) with QLoRA**.
**Runs on Colab (free T4 16 GB) / GPU with 12 GB or more.** Reproduces §6–§9 of `lora_qlora.md` by hand.

Flow: 4-bit load → attach LoRA → SFT training → inference comparison → save adapter → merge.

> This notebook requires a GPU. For concept-only exploration on CPU/Jetson, start with `lora_qlora_demo.ipynb`.

## 0. Environment Setup

In [ ]:
# Pin versions for API compatibility. Run on Colab; adjust for local installs.
%pip install -q -U "transformers>=4.44" "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.43" "datasets>=2.20" "accelerate>=0.33"

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU required (on Colab: Runtime -> GPU)'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
# Check if bf16 is available (Ampere and later). T4 uses fp16.
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('compute_dtype =', compute_dtype)

---

## 1. Load Base Model in 4-bit (NF4)

Settings from `lora_qlora.md` §5–§6. `nf4 + double_quant + compute_dtype` is the standard QLoRA configuration.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'   # ungated, lightweight. Qwen/Qwen2.5-0.5B-Instruct also works

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',            # NormalFloat4 (§5.2)
    bnb_4bit_use_double_quant=True,       # Double Quantization (§5.3)
    bnb_4bit_compute_dtype=compute_dtype, # compute in 16-bit (§6.1)
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map='auto',
)
model.config.use_cache = False  # off during training (compatible with gradient checkpointing)
print('VRAM after 4-bit load:', round(torch.cuda.memory_allocated()/1e9,2), 'GB')

---

## 2. Attach LoRA

Procedure from §7.2. Don't forget `prepare_model_for_kbit_training` (§11-1).

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)  # required preparation for training quantized models

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,                 # alpha/r = 2 (§2.4)
    target_modules='all-linear',   # all Linear layers (QLoRA paper recommendation, §3)
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# e.g.: trainable params: ~12M || all params: ~1.1B || trainable%: ~1.1%

---

## 3. Dataset

The standard QLoRA demo dataset `mlabonne/guanaco-llama2-1k` (1000 chat samples). Use the `text` column directly for SFT.

In [ ]:
from datasets import load_dataset
dataset = load_dataset('mlabonne/guanaco-llama2-1k', split='train')
print(dataset)
print('--- Sample ---')
print(dataset[0]['text'][:500])

---

## 4. Training (SFT)

`trl` SFTTrainer. Learning rate is higher than Full FT at 2e-4 (§8). Optimizer is paged 8-bit Adam (§5.4).
Only 60 steps for the demo (use `num_train_epochs` for production).

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir='./qlora-tinyllama',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch=8
    max_steps=60,                       # demo only; use num_train_epochs=1~3 for production
    learning_rate=2e-4,                 # LoRA can use a higher rate (§8)
    bf16=(compute_dtype==torch.bfloat16),
    fp16=(compute_dtype==torch.float16),
    logging_steps=10,
    optim='paged_adamw_8bit',           # paged optimizer (§5.4)
    gradient_checkpointing=True,        # reduce activation memory (§11-3)
    max_seq_length=512,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=args,
    processing_class=tokenizer,
)
trainer.train()
print('Peak VRAM during training:', round(torch.cuda.max_memory_allocated()/1e9,2), 'GB')

---

## 5. Verify with Inference

Generate output with the trained adapter attached.

In [ ]:
def generate(prompt, max_new_tokens=200):
    text = f'<s>[INST] {prompt} [/INST]'
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    model.config.use_cache = True
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=True, temperature=0.7, top_p=0.9)
    model.config.use_cache = False
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(generate('What is LoRA in machine learning? Answer concisely.'))

---

## 6. Save the Adapter

§9. Only the **adapter** is saved (tens of MB). The base model is managed separately.

In [ ]:
model.save_pretrained('./qlora-tinyllama-adapter')
tokenizer.save_pretrained('./qlora-tinyllama-adapter')
import os
sz = sum(os.path.getsize(os.path.join('./qlora-tinyllama-adapter',f))
         for f in os.listdir('./qlora-tinyllama-adapter'))
print('Saved size:', round(sz/1e6,1), 'MB (just the adapter, compared to the 1.1B base)')

---

## 7. Merge into a Standalone Model (Optional — for production deployment)

§9.1. **Cannot merge while still in 4-bit** → reload base in fp16, then merge safely.
(Skip if VRAM is insufficient)

In [ ]:
# Merge procedure for zero inference latency (watch VRAM)
from peft import PeftModel

base_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base_fp16, './qlora-tinyllama-adapter')
merged = merged.merge_and_unload()   # bake in W = W0 + (alpha/r) B A
merged.save_pretrained('./qlora-tinyllama-merged')
tokenizer.save_pretrained('./qlora-tinyllama-merged')
print('Merge complete -> ./qlora-tinyllama-merged (LoRA branch removed, zero added latency)')

---

## 8. Hands-On Exercises

Try these to deepen your understanding:

1. **Change r**: Compare `print_trainable_parameters()` and final loss for `r=8` vs `r=64`.
2. **Restrict target_modules**: Use only `['q_proj','v_proj']` and observe the difference in performance and parameter count (§3).
3. **Turn off quantization**: Compare VRAM against plain LoRA (`load_in_4bit=False`) (§4 memory).
4. **Effect of alpha**: Change `lora_alpha` to 16 / 64 and observe changes in generation quality.
5. **Try DoRA**: Set `LoraConfig(..., use_dora=True)` and compare (§10).
6. **Your own data**: Swap in your own dataset with a `text` column for domain adaptation.

> Log what you observe in #daily-action-log to have it reflected in the daily review's "Learning & Growth" section.